# Summarize from Different Documents

Combines a Wikipedia article with a local Word document, PDF, and text file about the same topic (Paestum), then uses Claude to produce a single refined summary across all sources.

## Setup
Imports LangChain building blocks (Anthropic chat model, token-based text splitter, prompt templates, output parser, runnables) and loads environment variables (e.g. `ANTHROPIC_API_KEY`) from a `.env` file.

In [ ]:
from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic
from langchain_text_splitters import TokenTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel
from langchain_community.document_loaders import WikipediaLoader
from langchain_community.document_loaders import Docx2txtLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import TextLoader
import wikipedia
import getpass

load_dotenv()

## Load a Wikipedia article
Searches Wikipedia for "Paestum" and loads up to 2 matching articles as LangChain `Document` objects. A custom User-Agent is set on the underlying `wikipedia` client to avoid getting rate-limited by the Wikimedia API.

In [ ]:
wikipedia.set_user_agent("summarize-from-different-documents/1.0 (amine.bahlouli1993@gmail.com)")

wikipedia_loader = WikipediaLoader(query="Paestum", load_max_docs=2)
wikipedia_docs = wikipedia_loader.load()


## Load local documents
Loads three local files about Paestum in different formats — a Word document, a PDF, and a plain text file — from the `paestrum/` folder, each parsed into `Document` objects via the matching LangChain loader (`Docx2txtLoader`, `PyPDFLoader`, `TextLoader`).

In [ ]:
word_loader = Docx2txtLoader("paestrum/Paestum-Britannica.docx")
word_docs = word_loader.load()

pdf_loader = PyPDFLoader("paestrum/PaestumRevisited.pdf")
pdf_docs = pdf_loader.load()

txt_loader = TextLoader("paestrum/Paestum-Encyclopedia.txt")
txt_docs = txt_loader.load()

## Combine all sources
Concatenates the Wikipedia, Word, PDF, and text documents into a single list (`all_docs`), so they can be summarized together regardless of their original format or source.

In [4]:
all_docs = wikipedia_docs + word_docs + pdf_docs + txt_docs

## Instantiate the LLM
Creates the Claude Haiku 4.5 chat model that will be used to generate summaries.

In [5]:
llm = ChatAnthropic(model="claude-haiku-4-5")

## Per-document summary prompt
Defines a prompt that asks the LLM to write a concise summary of a single document's text, wired into `doc_summary_chain` (prompt → LLM). Note: this chain is defined but not actually invoked below — the refine chain is used instead.

In [7]:
doc_summary_template = """Write a concise summary of the following text:
{text}
DOC SUMMARY:"""
doc_summary_prompt = PromptTemplate.from_template(doc_summary_template)

doc_summary_chain = doc_summary_prompt | llm

## Refine prompt
Defines the "refine" prompt: given the running summary produced so far and the text of one additional document, the LLM either folds the new content into an updated summary or leaves the existing summary unchanged if the new document adds nothing useful. Wired into `refine_chain` (prompt → LLM → string output parser).

In [8]:
refine_summary_template = """
You must produce a final summary from the current refined summary
which has been generated so far and from the content of an 
additional document.
This is the current refined summary generated so far:
{current_refined_summary}
This is the content of the additional document: {text}
Only use the content of the additional document if it is useful, 
otherwise return the current full summary as it is."""

refine_summary_prompt = PromptTemplate.from_template(refine_summary_template)

refine_chain = refine_summary_prompt | llm | StrOutputParser()

## Run the refine loop
`refine_summary` iterates over every document in `all_docs`, calling `refine_chain` once per document to progressively fold each one into a running summary. Returns the `final_summary` plus the list of `intermediate_steps` (inputs used at each iteration), stored in `full_summary`.

In [ ]:
def refine_summary(docs):

    intermediate_steps = []
    current_refined_summary = ''
    for doc in docs:
        intermediate_step = \
           {"current_refined_summary": current_refined_summary, 
            "text": doc.page_content}
        intermediate_steps.append(intermediate_step)

        current_refined_summary = refine_chain.invoke(intermediate_step)

    return {"final_summary": current_refined_summary,
            "intermediate_steps": intermediate_steps}

full_summary = refine_summary(all_docs)